[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harshini-Suresha/patient-drug-analysis/blob/main/drug_review_analytics.ipynb)
[![View site](https://img.shields.io/badge/site-GitHub_Pages-green)](https://harshini-suresha.github.io/patient-drug-analysis/)
[![Streamlit](https://img.shields.io/badge/app-Streamlit-red)](https://share.streamlit.io/)


# Patient-Reported Drug Review Analytics & Ranking
**Condition-stratified drug ranking using shrinkage estimation and interactive analytics**

Companion notebook to the website. It reproduces every number on the site from the raw UCI Drug Review Dataset (Drugs.com), and adds two analyses that go beyond the site's static views: a **Love-or-Hate polarisation index** and **bootstrap rank robustness**.

Run all cells in Google Colab (Runtime → Run all). If the automatic download is blocked, upload `archive.zip` (or the two `drugsCom*_raw` files) when prompted.

> Educational analysis of patient opinions. Not clinical evidence and not medical advice.

In [ ]:
import glob, html, re, urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
GREEN, TEAL, ROSE, AMBER = "#17703f", "#2fa566", "#d6456b", "#c98a00"
rng = np.random.default_rng(42)

## 1. Load the real data

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/462/drug+review+dataset+drugs+com.zip"

def find_files():
    tr = glob.glob("**/drugsComTrain_raw.*", recursive=True)
    te = glob.glob("**/drugsComTest_raw.*", recursive=True)
    return (tr[0], te[0]) if tr and te else None

files = find_files()
if not files:
    try:
        urllib.request.urlretrieve(URL, "uci.zip")
        zipfile.ZipFile("uci.zip").extractall("data")
        files = find_files()
    except Exception as e:
        print("Automatic download failed:", e)
if not files:
    from google.colab import files as colab_files      # upload archive.zip or the two files
    for name in colab_files.upload():
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall("data")
    files = find_files()

read = lambda p: pd.read_csv(p, sep="\t" if p.endswith(".tsv") else ",")
raw = pd.concat([read(f) for f in files], ignore_index=True)
print(f"{len(raw):,} raw reviews from {files}")
raw.head(3)

## 2. Clean and normalise
* drop rows with no condition, and rows where scraping leaked *"users found this comment helpful"* into the condition column
* title-case conditions, decode HTML entities in reviews
* repair condition labels that the source truncated (e.g. `Bipolar Disorde` → `Bipolar Disorder`), only where the original is unambiguous and the repaired label does not already exist

In [ ]:
FIX = {"Ibromyalgia":"Fibromyalgia","Emale Infertility":"Female Infertility","Atigue":"Fatigue",
 "Acial Wrinkles":"Facial Wrinkles","Ibrocystic Breast Disease":"Fibrocystic Breast Disease",
 "Amilial Mediterranean Feve":"Familial Mediterranean Fever","Ailure To Thrive":"Failure To Thrive",
 "Llicle Stimulation":"Follicle Stimulation","Llicular Lymphoma":"Follicular Lymphoma",
 "Ungal Infection Prophylaxis":"Fungal Infection Prophylaxis","Ungal Pneumonia":"Fungal Pneumonia",
 "Unctional Gastric Disorde":"Functional Gastric Disorder","Cal Segmental Glomerulosclerosis":"Focal Segmental Glomerulosclerosis",
 "Eve":"Fever","Zen Shoulde":"Frozen Shoulder","Mance Anxiety":"Performance Anxiety",
 "Not Listed / Othe":"Not Listed / Other","Premature Lab":"Premature Labor","Vertig":"Vertigo",
 "Impetig":"Impetigo","Tinea Versicol":"Tinea Versicolor","Cance":"Cancer"}
SUFFIX = [(r" Disorde$"," Disorder"),(r" Cance$"," Cancer"),(r" Tum$"," Tumor"),(r" Ulce$"," Ulcer"),
          (r" Bladde$"," Bladder"),(r" Zoste$"," Zoster"),(r" Trem$"," Tremor"),(r" Feve$"," Fever")]

artifact = raw["condition"].str.contains("users found this comment helpful", case=False, na=False)
df = raw.dropna(subset=["condition", "drugName", "rating"])
df = df[~df["condition"].str.contains("users found this comment helpful", case=False, na=False)].copy()
df["condition"] = df["condition"].str.strip().str.title().str.replace(r"'S\b", "'s", regex=True)
df["drugName"] = df["drugName"].str.strip()

have = set(df["condition"])
def repair(n):
    m = FIX.get(n, n)
    if m == n:
        for pat, rep in SUFFIX:
            m = re.sub(pat, rep, m)
    return n if (m != n and m in have) else m
mapping = {n: repair(n) for n in have}
df["condition"] = df["condition"].map(mapping)

df["review"] = df["review"].fillna("").map(html.unescape).str.strip('" ')
df["r"] = df["rating"].round().astype(int)
df["year"] = pd.to_datetime(df["date"], format="%d-%b-%y", errors="coerce").dt.year

print(f"raw {len(raw):,} | scraping artifacts removed {artifact.sum():,} | clean {len(df):,}")
print(f"conditions {df.condition.nunique():,} | drugs {df.drugName.nunique():,}")
print("labels repaired:", sum(a != b for a, b in mapping.items()))

## 3. The shape of the data

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
d = df["r"].value_counts().sort_index()
ax[0].bar(d.index, d.values, color=TEAL); ax[0].set_title("Rating distribution (U-shaped)"); ax[0].set_xlabel("rating")
y = df.groupby("year").agg(n=("rating", "size"), mean=("rating", "mean"))
ax[1].bar(y.index, y.n, color=GREEN); ax[1].set_title("Reviews per year"); 
ax[2].plot(y.index, y["mean"], marker="o", color=ROSE); ax[2].set_title("Mean rating per year")
plt.tight_layout(); plt.show()
print(f"mean rating {y['mean'].iloc[0]:.2f} ({y.index[0]}) -> {y['mean'].iloc[-1]:.2f} ({y.index[-1]})")
print(f"volume 2013 -> 2016: {y.loc[2013,'n']:,} -> {y.loc[2016,'n']:,}  ({y.loc[2016,'n']/y.loc[2013,'n']:.1f}x)")

In [ ]:
# Rating profile of the 15 most reviewed conditions (share of each rating)
top = df["condition"].value_counts().head(15).index
prof = pd.crosstab(df[df.condition.isin(top)]["condition"], df["r"], normalize="index").loc[top]
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(prof.values, cmap="Greens", aspect="auto")
ax.set_xticks(range(10)); ax.set_xticklabels(range(1, 11)); ax.set_yticks(range(len(top))); ax.set_yticklabels(top)
for i in range(prof.shape[0]):
    for j in range(10):
        ax.text(j, i, f"{prof.values[i,j]*100:.0f}", ha="center", va="center", fontsize=7)
ax.set_title("Rating profile by condition (% of reviews)"); plt.colorbar(im); plt.tight_layout(); plt.show()

## 4. Shrinkage ranking
For condition *c* and drug *d* with *n* reviews and mean *x̄*, the score is

`score = n/(n+m) · x̄ + m/(n+m) · μ_c`

where μ_c is the condition's mean rating and **m** is the prior strength (an empirical-Bayes style normal–normal posterior mean with fixed m).

In [ ]:
pairs = (df.groupby(["condition", "drugName"])
           .agg(n=("rating", "size"), mean=("rating", "mean"), useful=("usefulCount", "mean")).reset_index())
pairs["mu"] = pairs["condition"].map(df.groupby("condition")["rating"].mean())
print(f"{len(pairs):,} condition-drug pairs")

def score(p, m=10):
    return p.n / (p.n + m) * p["mean"] + m / (p.n + m) * p.mu

def rank_condition(cond, m=10, min_reviews=10):
    s = pairs[(pairs.condition == cond) & (pairs.n >= min_reviews)].copy()
    s["score"] = score(s, m)
    s["rank_raw"] = s["mean"].rank(ascending=False, method="min").astype(int)
    s["rank_shrunk"] = s["score"].rank(ascending=False, method="min").astype(int)
    return s.sort_values("score", ascending=False)

rank_condition("Depression").head(10)[["drugName", "n", "mean", "score", "rank_raw", "rank_shrunk"]].round(2)

In [ ]:
# Reliability funnel: small samples scatter widely around the overall mean
mu, sd = df.rating.mean(), df.rating.std()
n = np.logspace(0, np.log10(pairs.n.max()), 100)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(pairs.n, pairs["mean"], s=5, alpha=.25, color=TEAL)
ax.plot(n, mu + 1.96 * sd / np.sqrt(n), color=ROSE); ax.plot(n, mu - 1.96 * sd / np.sqrt(n), color=ROSE)
ax.axhline(mu, ls="--", color=GREEN); ax.set_xscale("log"); ax.set_ylim(1, 10)
ax.set_xlabel("reviews per pair (log)"); ax.set_ylabel("mean rating"); ax.set_title("Reliability funnel"); plt.show()

In [ ]:
# Sensitivity: how the top drugs' scores move as m grows
cond = "Depression"
s = pairs[(pairs.condition == cond) & (pairs.n >= 10)]
top6 = s.sort_values("mean", ascending=False).head(6)
ms = [0, 1, 2, 5, 10, 20, 35, 50, 75, 100]
fig, ax = plt.subplots(figsize=(8, 4.5))
for _, r in top6.iterrows():
    ax.plot(ms, [r["mean"] if m == 0 else score(r, m) for m in ms], marker="o", label=f"{r.drugName} (n={r.n})")
ax.set_xlabel("shrinkage m"); ax.set_ylabel("score"); ax.set_title(f"Shrinkage sensitivity: {cond}"); ax.legend(fontsize=7); plt.show()

## 5. Validation: are shrunk rankings more reproducible? (split-half)

In [ ]:
def split_half(m_values=(0, 1, 5, 10, 25, 50), repeats=10, min_cond=300, min_drug=10):
    res = {m: [] for m in m_values}; used = 0
    sizes = df.groupby("condition").size()
    for c in sizes.index[sizes >= min_cond]:
        s = df[df.condition == c]; cnt = s.groupby("drugName").size(); keep = cnt[cnt >= min_drug].index
        if len(keep) < 6: continue
        used += 1; s = s[s.drugName.isin(keep)]; v = s.rating.values; dr = s.drugName.values
        for _ in range(repeats):
            msk = rng.random(len(s)) < .5; halves = []
            for h in (msk, ~msk):
                g = pd.DataFrame({"d": dr[h], "r": v[h]}).groupby("d").r.agg(["mean", "size"]); halves.append((g, v[h].mean()))
            idx = halves[0][0].index.intersection(halves[1][0].index)
            if len(idx) < 5: continue
            for m in m_values:
                sc = []
                for g, C in halves:
                    q = g.loc[idx]; sc.append(q["mean"] if m == 0 else q["size"]/(q["size"]+m)*q["mean"] + m/(q["size"]+m)*C)
                r = spearmanr(sc[0], sc[1]).correlation
                if not np.isnan(r): res[m].append(r)
    return used, pd.Series({m: np.mean(x) for m, x in res.items()}, name="mean split-half Spearman rho")

used, stab = split_half()
print(f"{used} conditions")
stab.round(3)

In [ ]:
stab.plot.bar(color=TEAL, figsize=(6, 3.5), rot=0); plt.ylim(.4, stab.max() + .05)
plt.xlabel("shrinkage m (0 = raw mean)"); plt.title("Ranking reproducibility vs shrinkage"); plt.show()

## 6. Something new: the **Love-or-Hate Index**
Averages hide polarisation. For each drug define `LHI = 4 · p_low · p_high`, where `p_low` is the share of ratings ≤ 3 and `p_high` the share ≥ 8. LHI is 0 when reviewers agree and reaches 1 when they split 50/50 between loving and hating it. A drug with a mediocre mean *and* a high LHI is not mediocre, it is divisive.

In [ ]:
g = df.groupby("drugName").agg(n=("r", "size"), mean=("rating", "mean"),
                              p_low=("r", lambda x: (x <= 3).mean()), p_high=("r", lambda x: (x >= 8).mean()))
g["LHI"] = 4 * g.p_low * g.p_high
big = g[g.n >= 100]
print("Most divisive drugs (>=100 reviews):")
display(big.sort_values("LHI", ascending=False).head(10).round(2))
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(big["mean"], big.LHI, s=np.sqrt(big.n) * 2, alpha=.45, c=big.LHI, cmap="RdYlGn_r")
ax.set_xlabel("mean rating"); ax.set_ylabel("Love-or-Hate Index"); ax.set_title("Divisive drugs sit high on the index even with mid-range means")
plt.show()

## 7. Something new: **rank robustness** by bootstrap
A rank is a point estimate. Resampling each drug's reviews with replacement (B = 500), re-scoring and re-ranking gives the **probability a drug is truly #1 or top-3**, and a 90% rank interval.

In [ ]:
def rank_robustness(cond, m=10, min_reviews=30, B=500):
    s = df[df.condition == cond]; C = s.rating.mean()
    arrs = {d: x.rating.values for d, x in s.groupby("drugName") if len(x) >= min_reviews}
    names = list(arrs)
    scores = np.empty((B, len(names)))
    for j, d in enumerate(names):
        a = arrs[d]; n = len(a)
        means = rng.choice(a, size=(B, n)).mean(axis=1)
        scores[:, j] = n/(n+m)*means + m/(n+m)*C
    ranks = (-scores).argsort(axis=1).argsort(axis=1) + 1
    out = pd.DataFrame({"drug": names, "n": [len(arrs[d]) for d in names],
                        "P(rank 1)": (ranks == 1).mean(0), "P(top 3)": (ranks <= 3).mean(0),
                        "rank 5%": np.percentile(ranks, 5, axis=0), "rank 95%": np.percentile(ranks, 95, axis=0)})
    return out.sort_values("P(top 3)", ascending=False)

rr = rank_robustness("Depression")
display(rr.head(10).round(2))
t = rr.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hlines(range(len(t)), t["rank 5%"], t["rank 95%"], color=ROSE, lw=3)
ax.set_yticks(range(len(t))); ax.set_yticklabels(t.drug); ax.set_xlabel("rank (1 = best), 90% bootstrap interval")
ax.set_title("How certain are the top ranks? Depression"); plt.tight_layout(); plt.show()

## 7b. Something new: **patient language**
What do patients write when they love or hate a drug? Words are compared between 8-10 and 1-3 ratings with a log-odds z-score and an informative Dirichlet prior (Monroe et al., 2008), so frequent words are not over-weighted.

In [ ]:
from collections import Counter
STOP = set("""a about above after again all also am an and any are as at be because been before being below between both but by can could did do does doing down during each few for from further get got had has have having he her here hers him his how i if in into is it its just me more most my no nor not now of off on once only or other our out over own same she should so some such than that the their them then there these they this those through to too under until up very was we were what when where which while who whom why will with would you your yours im ive dont didnt cant wont doesnt isnt wasnt one two like even really much still back going take took taking day days time week weeks month months year years first last long ever lot since another around make made thing things say said told though""".split())
tok = re.compile(r"[a-z]{3,}")
text = df["review"].str.lower()
low, high, allc = Counter(), Counter(), Counter()
for rev, r in zip(text.values, df["r"].values):
    ws = [w for w in tok.findall(rev) if w not in STOP]
    allc.update(ws)
    if r <= 3: low.update(ws)
    elif r >= 8: high.update(ws)
n1, n2, N, a0 = sum(low.values()), sum(high.values()), sum(allc.values()), 2000
rows = []
for w, c in allc.items():
    if c < 400: continue
    a = a0 * c / N; y1, y2 = low[w], high[w]
    d = np.log((y1+a)/(n1+a0-y1-a)) - np.log((y2+a)/(n2+a0-y2-a))
    rows.append((w, d / np.sqrt(1/(y1+a) + 1/(y2+a))))
rows.sort(key=lambda x: x[1])
love, hate = rows[:14], rows[::-1][:14]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].barh([w for w, _ in love][::-1], [-z for _, z in love][::-1], color=TEAL); ax[0].set_title("Words that mark 8-10 ratings")
ax[1].barh([w for w, _ in hate][::-1], [z for _, z in hate][::-1], color=ROSE); ax[1].set_title("Words that mark 1-3 ratings")
plt.tight_layout(); plt.show()

In [ ]:
pats = {"side effects (any)": r"side effect", "weight gain": r"weight gain|gained weight|gaining weight", "nausea": r"nause",
        "headache": r"headache", "dizziness": r"dizz", "fatigue / drowsy": r"fatigue|tired|drows|sleepy|lethargic",
        "rash / itching": r"\brash|hives|itch", "bleeding / spotting": r"bleed|spotting",
        "worked / helped": r"worked|helped", "stopped taking": r"stopped taking|quit taking|discontinu"}
band = pd.cut(df["r"], [0, 3, 7, 10], labels=["1-3", "4-7", "8-10"])
se = pd.DataFrame({k: [text.str.contains(p, regex=True)[band == b].mean() * 100 for b in ["1-3", "4-7", "8-10"]]
                   for k, p in pats.items()}, index=["1-3", "4-7", "8-10"]).T
display(se.round(1))
se.plot.barh(figsize=(8, 5), color=[ROSE, "#9aa5a0", "#0072B2"]); plt.xlabel("% of reviews mentioning the term"); plt.show()
anat = df.assign(words=text.str.split().str.len()).groupby("r").agg(helpful=("usefulCount", "mean"), words=("words", "median"))
display(anat.round(1))

## 7c. Something new: **head-to-head** comparison
Is drug A really better than drug B? With the same normal-normal model as the shrinkage score (posterior mean = shrunk score, posterior variance = sigma^2/(n+m)), `P(A > B)` is the probability that A's true mean rating is higher.

In [ ]:
from scipy.stats import norm

def head_to_head(cond, drug_a, drug_b, m=10):
    s = df[df.condition == cond]; C = s.rating.mean(); out = {}
    for name in (drug_a, drug_b):
        x = s[s.drugName == name].rating.values; n = len(x)
        out[name] = (n/(n+m)*x.mean() + m/(n+m)*C, x.var(ddof=0)/(n+m), n)
    (ma, va, na), (mb, vb, nb) = out[drug_a], out[drug_b]
    return {"shrunk A": ma, "shrunk B": mb, "n A": na, "n B": nb, "P(A > B)": norm.cdf((ma-mb)/np.sqrt(va+vb))}

top2 = rank_condition("Depression").head(2).drugName.tolist()
print(top2, head_to_head("Depression", *top2))

## 8. Export

In [ ]:
out = pairs.copy(); out["score_m10"] = score(out, 10)
out.round(4).to_csv("condition_drug_scores.csv", index=False)
print(len(out), "rows written to condition_drug_scores.csv")
try:
    from google.colab import files as colab_files; colab_files.download("condition_drug_scores.csv")
except Exception:
    pass

## Limits
No causal identification; self-selected reviewers; unobserved dose, severity and duration; possible duplicate or fabricated reviews; label noise from scraping (partly repaired above); and temporal drift in the reviewer population. The shrinkage strength *m* is fixed, not fitted. Treat all rankings as descriptive.